In [ ]:
import catboost as cb
# Check if GPU is available for CatBoost
print("\n=== Checking CatBoost GPU Support ===")
try:
    gpu_params = {'task_type': 'GPU', 'devices': '0'}
    test_model = cb.CatBoost(gpu_params)
    print("GPU is available for CatBoost!")
    has_gpu = True
except Exception as e:
    print(f"GPU is not available: {e}")
    print("Falling back to CPU.")
    has_gpu = False


=== Checking CatBoost GPU Support ===
GPU is available for CatBoost!


: 

In [1]:
import pandas as pd
import numpy as np
import catboost as cb
import matplotlib.pyplot as plt
import time
import optuna
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# Load the dataset
df = pd.read_csv('bus_eta_standard_scaled.csv')

print("Data shape:", df.shape)

# Prepare the data
X = df.drop(columns=['timestamp', 'eta_minutes', 'stop_sequence'])  # Features
y = df['eta_minutes']  # Target

# Print feature information
print("\n=== Feature and Target Information ===")
print(f"Target variable: 'eta_minutes'")
print(f"Number of features: {X.shape[1]}")
print("\nFeature list:")
for i, feature in enumerate(X.columns):
    print(f"{i+1}. {feature}")

# Function to evaluate and visualize model performance
def evaluate_model(model, X_test, y_test, model_name):
    # Make predictions
    y_pred = model.predict(X_test)
    
    # Calculate metrics
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
    accuracy = 100 - mape
    
    print(f"--- {model_name} Performance Metrics ---")
    print(f'RMSE: {rmse:.2f}')
    print(f'R² Score: {r2:.4f}')
    print(f"MAPE: {mape:.2f}%")
    print(f"Prediction Accuracy: {accuracy:.2f}%")
    
    # Create visualizations
    fig, axes = plt.subplots(2, 2, figsize=(20, 15))
    
    # Scatter plot of actual vs predicted
    axes[0, 0].scatter(y_test, y_pred, alpha=0.5)
    axes[0, 0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r', linestyle='--')
    axes[0, 0].set_xlabel("Actual ETA")
    axes[0, 0].set_ylabel("Predicted ETA")
    axes[0, 0].set_title(f"{model_name}: Actual vs. Predicted ETA")
    
    # Line plot of first 50 samples
    axes[0, 1].plot(y_test.values[:50], label="Actual", marker='o')
    axes[0, 1].plot(y_pred[:50], label="Predicted", marker='x')
    axes[0, 1].set_xlabel("Sample Index")
    axes[0, 1].set_ylabel("ETA (minutes)")
    axes[0, 1].set_title(f"{model_name}: Actual vs. Predicted ETA (First 50 Samples)")
    axes[0, 1].legend()
    
    # Histogram of errors
    errors = y_pred - y_test
    axes[1, 0].hist(errors, bins=30, edgecolor='black')
    axes[1, 0].set_xlabel("Prediction Error (Predicted - Actual)")
    axes[1, 0].set_ylabel("Frequency")
    axes[1, 0].set_title(f"{model_name}: Histogram of Prediction Errors")
    
    # Percentage error vs actual
    percentage_error = 100 * (y_pred - y_test) / y_test
    axes[1, 1].scatter(y_test, percentage_error, alpha=0.5)
    axes[1, 1].axhline(y=0, color='r', linestyle='--')
    axes[1, 1].set_xlabel("Actual ETA (minutes)")
    axes[1, 1].set_ylabel("Prediction Error (%)")
    axes[1, 1].set_title(f"{model_name}: Prediction Error vs. Actual ETA")
    
    plt.tight_layout()
    plt.show()
    
    return {
        'rmse': rmse,
        'r2': r2,
        'mape': mape,
        'accuracy': accuracy,
        'predictions': y_pred
    }

# Splitting the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("\nTraining set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

# ============== OPTUNA OPTIMIZATION FOR CATBOOST WITH GPU ==============

def objective(trial):
    # Define boosting_type
    boosting_type = trial.suggest_categorical('boosting_type', ['Ordered', 'Plain'])
    
    # Choose grow_policy based on boosting_type
    if boosting_type == 'Ordered':
        grow_policy = 'SymmetricTree'  # Only option for Ordered
    else:
        grow_policy = trial.suggest_categorical('grow_policy', ['SymmetricTree', 'Depthwise', 'Lossguide'])
    
    bootstrap_type = trial.suggest_categorical('bootstrap_type', ['Bayesian', 'Bernoulli', 'MVS'])
    
    param = {
        'iterations': trial.suggest_int('iterations', 500, 10000),
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.3, log=True),
        'depth': trial.suggest_int('depth', 4, 16),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-8, 100.0, log=True),
        'random_strength': trial.suggest_float('random_strength', 1e-8, 10.0, log=True),
        'border_count': trial.suggest_int('border_count', 32, 255),  # Keep this
        'grow_policy': grow_policy,
        'boosting_type': boosting_type,
        'bootstrap_type': bootstrap_type,
        'loss_function': 'RMSE',
        'eval_metric': 'RMSE',
        'verbose': 0,
        'task_type': 'GPU',
        'devices': '0',
        'gpu_ram_part': 0.95,  # Use 95% of GPU memory
        'use_best_model': True,
        'thread_count': -1,     # Use all available CPU cores for data processing
        'gpu_cat_features_storage': 'CpuPinnedMemory',  # Store categorical features in pinned memory for faster GPU transfer
        # Removed scale_pos_weight parameter
        'boosting_type': boosting_type,
        'bootstrap_type': bootstrap_type,
        # 'max_bin': 254,  # Remove this line to avoid conflict with border_count
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 100) if grow_policy in ['Lossguide', 'Depthwise'] else 1,
    }
    
    # Add bagging_temperature only for Bayesian bootstrap
    if param['bootstrap_type'] == 'Bayesian':
        param['bagging_temperature'] = trial.suggest_float('bagging_temperature', 0.0, 10.0)
    
    # Only add max_leaves for Lossguide
    if param['grow_policy'] == 'Lossguide':
        param['max_leaves'] = trial.suggest_int('max_leaves', 31, 1000)
    
    # Add subsample only for Bernoulli bootstrap
    if param['bootstrap_type'] == 'Bernoulli':
        param['subsample'] = trial.suggest_float('subsample', 0.5, 1.0)
    
    # Create dataset
    train_pool = cb.Pool(X_train, label=y_train)
    eval_pool = cb.Pool(X_test, label=y_test)
    
    # Train model with early stopping
    model = cb.CatBoost(param)
    model.fit(
        train_pool,
        eval_set=eval_pool,
        early_stopping_rounds=100,
        verbose=0
    )
    
    # Return the validation error
    return model.get_best_score()['validation']['RMSE']

print("\n=== Starting Optuna Hyperparameter Optimization for CatBoost with GPU ===")
start_time = time.time()

# Create a study and optimize
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)  # Increase for better results, but longer runtime

optimization_time = time.time() - start_time
print(f"Optimization completed in {optimization_time:.2f} seconds ({optimization_time/60:.2f} minutes)")

# Get the best parameters
best_params = study.best_params
best_params['loss_function'] = 'RMSE'
best_params['eval_metric'] = 'RMSE'
best_params['verbose'] = 100
best_params['task_type'] = 'GPU'
best_params['devices'] = '0'
best_params['gpu_ram_part'] = 0.95  # Use 95% of GPU memory
best_params['thread_count'] = -1    # Use all CPU cores for data processing
best_params['gpu_cat_features_storage'] = 'CpuPinnedMemory'  # Better GPU memory utilization
best_params['use_best_model'] = True

# Make sure scale_pos_weight is not present
if 'scale_pos_weight' in best_params:
    best_params.pop('scale_pos_weight')

# Ensure compatibility between boosting_type and grow_policy
if best_params['boosting_type'] == 'Ordered' and best_params.get('grow_policy', 'SymmetricTree') != 'SymmetricTree':
    best_params['grow_policy'] = 'SymmetricTree'
    print("Fixed incompatible parameter combination: Ordered boosting requires SymmetricTree")

# If bootstrap_type='Bayesian', we must remove subsample parameter
if best_params['bootstrap_type'] == 'Bayesian' and 'subsample' in best_params:
    best_params.pop('subsample')

print("\n=== Best Hyperparameters ===")
for param_name, param_value in best_params.items():
    print(f"{param_name}: {param_value}")

# ============== TRAINING FINAL MODEL WITH GPU ==============

print("\n=== Training Final Optimized CatBoost Model on GPU ===")
start_time = time.time()

# Create dataset
train_pool = cb.Pool(X_train, label=y_train)
eval_pool = cb.Pool(X_test, label=y_test)

# Use the best parameters to train the final model
optimized_model = cb.CatBoost(best_params)
optimized_model.fit(
    train_pool,
    eval_set=eval_pool,
    early_stopping_rounds=100,
    verbose=200  # Print progress every 200 iterations
)

final_training_time = time.time() - start_time
print(f"Final model training time: {final_training_time:.2f} seconds ({final_training_time/60:.2f} minutes)")

# Evaluate the final model
final_results = evaluate_model(optimized_model, X_test, y_test, "Optuna Optimized CatBoost Model (GPU)")

# Feature importance plot
plt.figure(figsize=(14, 10))
feature_importances = optimized_model.get_feature_importance()
feature_names = X.columns
importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': feature_importances})
importance_df = importance_df.sort_values('Importance', ascending=False)

plt.barh(importance_df['Feature'][:30][::-1], importance_df['Importance'][:30][::-1])
plt.xlabel('Importance')
plt.title('Feature Importance (Optuna Optimized CatBoost Model - GPU)')
plt.tight_layout()
plt.show()

# Print the top 20 most important features
print("\n=== Top 20 Most Important Features ===")
for i, (feature, importance) in enumerate(zip(importance_df['Feature'][:20], importance_df['Importance'][:20])):
    print(f"{i+1}. {feature}: {importance}")

# Save the optimized model
model_file = 'catboost_optuna_optimized_model_gpu.cbm'
optimized_model.save_model(model_file)
print(f"Optimized model saved as '{model_file}'")

# Also save as joblib for easier loading
joblib_file = 'catboost_optuna_optimized_model_gpu.joblib'
joblib.dump(optimized_model, joblib_file)
print(f"Optimized model also saved as '{joblib_file}'")

# ============== TIME HORIZON PREDICTIONS ==============

print("\n=== Time Horizon Predictions ===")

# Define the time horizons in minutes
time_horizons = [1, 3, 5, 15, 30, 60]  # 1 min, 3 mins, 5 mins, 15 mins, 30 mins, 1 hour

# Function to filter test data for specific time horizons
def get_horizon_data(X_test, y_test, horizon, tolerance=0.5):
    """
    Filter the test data to include samples close to the specified time horizon.
    
    Args:
        X_test: Test features
        y_test: Test target values
        horizon: Target time horizon in minutes
        tolerance: Tolerance range (±) in minutes
        
    Returns:
        X_horizon, y_horizon: Filtered data for the specified horizon
    """
    # Convert to numpy arrays if they're not already
    if isinstance(X_test, pd.DataFrame):
        X_test = X_test.values
    if isinstance(y_test, pd.Series):
        y_test = y_test.values
    
    # Find indices where y_test is within the horizon ± tolerance
    horizon_mask = (y_test >= horizon - tolerance) & (y_test <= horizon + tolerance)
    
    if not horizon_mask.any():
        print(f"No exact data points found for horizon {horizon} minutes (±{tolerance})")
        # If no exact matches, get closest values
        distances = np.abs(y_test - horizon)
        closest_indices = np.argsort(distances)[:max(int(len(y_test) * 0.05), 10)]  # Get top 5% or at least 10 samples
        horizon_mask = np.zeros_like(y_test, dtype=bool)
        horizon_mask[closest_indices] = True
        print(f"Using {sum(horizon_mask)} closest data points instead")
    else:
        print(f"Found {sum(horizon_mask)} data points for horizon {horizon} minutes (±{tolerance})")
    
    X_horizon = X_test[horizon_mask]
    y_horizon = y_test[horizon_mask]
    
    return X_horizon, y_horizon

# Results storage
horizon_results = {}

# Test the model on each time horizon
print("\n=== Time Horizon Performance Comparison ===")
print(f"{'Time Horizon':<15} {'Samples':<10} {'RMSE':<10} {'R²':<10} {'MAPE':<10} {'Accuracy':<10}")
print("-" * 65)

for horizon in time_horizons:
    # Get data for this horizon
    X_horizon, y_horizon = get_horizon_data(X_test, y_test, horizon)
    
    if len(X_horizon) < 5:  # Skip if too few samples
        print(f"{horizon} min: Insufficient data")
        continue
    
    # Make predictions
    y_pred = optimized_model.predict(X_horizon)
    
    # Calculate metrics
    rmse = np.sqrt(mean_squared_error(y_horizon, y_pred))
    r2 = r2_score(y_horizon, y_pred)
    
    # Avoid division by zero in MAPE calculation
    valid_indices = y_horizon != 0
    if np.any(valid_indices):
        mape = np.mean(np.abs((y_horizon[valid_indices] - y_pred[valid_indices]) / y_horizon[valid_indices])) * 100
    else:
        mape = np.nan
    
    accuracy = 100 - mape if not np.isnan(mape) else np.nan
    
    horizon_results[horizon] = {
        'samples': len(X_horizon),
        'rmse': rmse,
        'r2': r2,
        'mape': mape,
        'accuracy': accuracy
    }
    
    print(f"{horizon} min:{'':<9} {len(X_horizon):<10d} {rmse:<10.2f} {r2:<10.4f} {mape:<10.2f}% {accuracy:<10.2f}%")

# Visualization of accuracy across time horizons
plt.figure(figsize=(12, 6))
horizons = list(horizon_results.keys())
accuracies = [horizon_results[h]['accuracy'] for h in horizons]
rmse_values = [horizon_results[h]['rmse'] for h in horizons]

fig, ax1 = plt.subplots(figsize=(12, 6))

color1 = 'tab:blue'
ax1.set_xlabel('Time Horizon (minutes)')
ax1.set_ylabel('Accuracy (%)', color=color1)
ax1.plot(horizons, accuracies, marker='o', color=color1, linestyle='-', linewidth=2)
ax1.tick_params(axis='y', labelcolor=color1)

ax2 = ax1.twinx()
color2 = 'tab:red'
ax2.set_ylabel('RMSE', color=color2)
ax2.plot(horizons, rmse_values, marker='s', color=color2, linestyle='--', linewidth=2)
ax2.tick_params(axis='y', labelcolor=color2)

plt.title('CatBoost Model Performance Across Different Time Horizons (GPU)')
plt.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

# ============== MAKING PREDICTIONS FOR SPECIFIC TIME WINDOWS ==============

print("\n=== Making Predictions for Specific Future Time Windows ===")

# Function to create a dataset for a specific prediction window
def create_prediction_window_dataset(X_test, y_test, window_start, window_end):
    """
    Create a dataset for predictions within a specific time window.
    
    Args:
        X_test: Test features
        y_test: Test target values
        window_start: Start time of the window (in minutes)
        window_end: End time of the window (in minutes)
        
    Returns:
        X_window, y_window: Data for the specified window
    """
    window_mask = (y_test >= window_start) & (y_test <= window_end)
    
    if not window_mask.any():
        print(f"No data points found for window {window_start}-{window_end} minutes")
        # If no exact matches, get closest values
        distances = np.minimum(np.abs(y_test - window_start), np.abs(y_test - window_end))
        closest_indices = np.argsort(distances)[:max(int(len(y_test) * 0.05), 10)]
        window_mask = np.zeros_like(y_test, dtype=bool)
        window_mask[closest_indices] = True
        print(f"Using {sum(window_mask)} closest data points instead")
    else:
        print(f"Found {sum(window_mask)} data points for window {window_start}-{window_end} minutes")
    
    X_window = X_test[window_mask]
    y_window = y_test[window_mask]
    
    return X_window, y_window

# Define time windows (in minutes)
time_windows = [
    (0, 2),     # 0-2 minutes
    (2, 5),     # 2-5 minutes
    (5, 15),    # 5-15 minutes
    (15, 30),   # 15-30 minutes
    (30, 60),   # 30-60 minutes
    (60, 120)   # 1-2 hours
]

# Results storage for time windows
window_results = {}

# Test the model on each time window
print("\n=== Time Window Performance Comparison ===")
print(f"{'Time Window':<15} {'Samples':<10} {'RMSE':<10} {'R²':<10} {'MAPE':<10} {'Accuracy':<10}")
print("-" * 65)

for window_start, window_end in time_windows:
    # Get data for this window
    X_window, y_window = create_prediction_window_dataset(X_test.values, y_test.values, window_start, window_end)
    
    if len(X_window) < 5:  # Skip if too few samples
        print(f"{window_start}-{window_end} min: Insufficient data")
        continue
    
    # Make predictions
    y_pred = optimized_model.predict(X_window)
    
    # Calculate metrics
    rmse = np.sqrt(mean_squared_error(y_window, y_pred))
    r2 = r2_score(y_window, y_pred)
    
    # Avoid division by zero in MAPE calculation
    valid_indices = y_window != 0
    if np.any(valid_indices):
        mape = np.mean(np.abs((y_window[valid_indices] - y_pred[valid_indices]) / y_window[valid_indices])) * 100
    else:
        mape = np.nan
        
    accuracy = 100 - mape if not np.isnan(mape) else np.nan
    
    window_name = f"{window_start}-{window_end} min"
    window_results[window_name] = {
        'samples': len(X_window),
        'rmse': rmse,
        'r2': r2,
        'mape': mape,
        'accuracy': accuracy
    }
    
    print(f"{window_name:<15} {len(X_window):<10d} {rmse:<10.2f} {r2:<10.4f} {mape:<10.2f}% {accuracy:<10.2f}%")

# Visualization of accuracy across time windows
plt.figure(figsize=(12, 6))
window_names = list(window_results.keys())
accuracies = [window_results[w]['accuracy'] for w in window_names]
rmse_values = [window_results[w]['rmse'] for w in window_names]

fig, ax1 = plt.subplots(figsize=(12, 6))

color1 = 'tab:blue'
ax1.set_xlabel('Time Window (minutes)')
ax1.set_ylabel('Accuracy (%)', color=color1)
ax1.plot(window_names, accuracies, marker='o', color=color1, linestyle='-', linewidth=2)
ax1.tick_params(axis='y', labelcolor=color1)
ax1.set_xticklabels(window_names, rotation=45)

ax2 = ax1.twinx()
color2 = 'tab:red'
ax2.set_ylabel('RMSE', color=color2)
ax2.plot(window_names, rmse_values, marker='s', color=color2, linestyle='--', linewidth=2)
ax2.tick_params(axis='y', labelcolor=color2)

plt.title('CatBoost Model Performance Across Different Time Windows (GPU)')
plt.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

# ============== COMPARISON WITH LIGHTGBM MODEL ==============

print("\n=== Comparing CatBoost with LightGBM Model (if available) ===")

try:
    # Try to load the LightGBM model if it exists
    lgb_model = joblib.load('lightgbm_optuna_optimized_model.joblib')
    
    # Make predictions
    lgb_predictions = lgb_model.predict(X_test)
    catboost_predictions = optimized_model.predict(X_test)
    
    # Calculate metrics for both models
    lgb_rmse = np.sqrt(mean_squared_error(y_test, lgb_predictions))
    lgb_r2 = r2_score(y_test, lgb_predictions)
    valid_indices = y_test != 0
    lgb_mape = np.mean(np.abs((y_test[valid_indices] - lgb_predictions[valid_indices]) / y_test[valid_indices])) * 100
    lgb_accuracy = 100 - lgb_mape
    
    catboost_rmse = np.sqrt(mean_squared_error(y_test, catboost_predictions))
    catboost_r2 = r2_score(y_test, catboost_predictions)
    catboost_mape = np.mean(np.abs((y_test[valid_indices] - catboost_predictions[valid_indices]) / y_test[valid_indices])) * 100
    catboost_accuracy = 100 - catboost_mape
    
    # Print comparison
    print("\n=== Model Comparison ===")
    print(f"{'Metric':<15} {'LightGBM':<15} {'CatBoost (GPU)':<15} {'Difference':<15}")
    print("-" * 60)
    print(f"{'RMSE':<15} {lgb_rmse:<15.2f} {catboost_rmse:<15.2f} {catboost_rmse - lgb_rmse:<15.2f}")
    print(f"{'R²':<15} {lgb_r2:<15.4f} {catboost_r2:<15.4f} {catboost_r2 - lgb_r2:<15.4f}")
    print(f"{'MAPE (%)':<15} {lgb_mape:<15.2f} {catboost_mape:<15.2f} {catboost_mape - lgb_mape:<15.2f}")
    print(f"{'Accuracy (%)':<15} {lgb_accuracy:<15.2f} {catboost_accuracy:<15.2f} {catboost_accuracy - lgb_accuracy:<15.2f}")
    
    # Visualize comparison
    fig, axes = plt.subplots(2, 2, figsize=(20, 15))
    
    # Actual vs. LightGBM vs. CatBoost prediction comparison for first 50 samples
    axes[0, 0].plot(y_test.values[:50], label="Actual", marker='o')
    axes[0, 0].plot(lgb_predictions[:50], label="LightGBM", marker='x')
    axes[0, 0].plot(catboost_predictions[:50], label="CatBoost (GPU)", marker='^')
    axes[0, 0].set_xlabel("Sample Index")
    axes[0, 0].set_ylabel("ETA (minutes)")
    axes[0, 0].set_title("Model Comparison: Actual vs. Predicted ETA (First 50 Samples)")
    axes[0, 0].legend()
    
    # Histogram of errors comparison
    lgb_errors = lgb_predictions - y_test
    catboost_errors = catboost_predictions - y_test
    axes[0, 1].hist(lgb_errors, bins=30, alpha=0.5, label="LightGBM", edgecolor='black')
    axes[0, 1].hist(catboost_errors, bins=30, alpha=0.5, label="CatBoost (GPU)", edgecolor='black')
    axes[0, 1].set_xlabel("Prediction Error (Predicted - Actual)")
    axes[0, 1].set_ylabel("Frequency")
    axes[0, 1].set_title("Model Comparison: Histogram of Prediction Errors")
    axes[0, 1].legend()
    
    # RMSE and Accuracy comparison
    models = ['LightGBM', 'CatBoost (GPU)']
    rmse_values = [lgb_rmse, catboost_rmse]
    accuracy_values = [lgb_accuracy, catboost_accuracy]
    
    bar_width = 0.35
    indices = np.arange(len(models))
    
    axes[1, 0].bar(indices, rmse_values, bar_width, label='RMSE')
    axes[1, 0].set_xlabel('Model')
    axes[1, 0].set_ylabel('RMSE')
    axes[1, 0].set_title('RMSE Comparison')
    axes[1, 0].set_xticks(indices)
    axes[1, 0].set_xticklabels(models)
    
    axes[1, 1].bar(indices, accuracy_values, bar_width, label='Accuracy (%)')
    axes[1, 1].set_xlabel('Model')
    axes[1, 1].set_ylabel('Accuracy (%)')
    axes[1, 1].set_title('Accuracy Comparison')
    axes[1, 1].set_xticks(indices)
    axes[1, 1].set_xticklabels(models)
    
    plt.tight_layout()
    plt.show()
    
except FileNotFoundError:
    print("LightGBM model not found. Skipping comparison.")

# ============== SAVE RESULTS TO CSV ==============

# Save time horizon results
horizon_df = pd.DataFrame.from_dict(horizon_results, orient='index')
horizon_df.index.name = 'horizon_minutes'
horizon_df.to_csv('catboost_gpu_time_horizon_results.csv')
print("\nTime horizon results saved to 'catboost_gpu_time_horizon_results.csv'")

# Save time window results
window_df = pd.DataFrame.from_dict(window_results, orient='index')
window_df.index.name = 'time_window'
window_df.to_csv('catboost_gpu_time_window_results.csv')
print("Time window results saved to 'catboost_gpu_time_window_results.csv'")

print("\n=== CatBoost GPU Optimization Complete ===")

# Load the model and make a prediction on a random sample
print("\n=== Example Prediction ===")
sample_idx = np.random.randint(0, len(X_test))
sample_data = X_test.iloc[[sample_idx]]
actual_value = y_test.iloc[sample_idx]
predicted_value = optimized_model.predict(sample_data)[0]
print(f"Sample {sample_idx}:")
print(f"Actual ETA: {actual_value:.2f} minutes")
print(f"Predicted ETA: {predicted_value:.2f} minutes")
print(f"Absolute Error: {abs(actual_value - predicted_value):.2f} minutes")
print(f"Percentage Error: {100 * abs(actual_value - predicted_value) / actual_value:.2f}%")

c:\Users\cheng\anaconda3\envs\GPU\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2025-03-24 04:01:07,511] A new study created in memory with name: no-name-61b490df-e093-4b36-9fc2-b64db39c0c0f


Data shape: (100000, 14)

=== Feature and Target Information ===
Target variable: 'eta_minutes'
Number of features: 11

Feature list:
1. current_stop_name
2. next_stop_name
3. day_of_week
4. is_holiday
5. is_peak_hour
6. weather_condition
7. passenger_count
8. current_speed
9. distance_to_next_stop
10. current_lat
11. current_lon

Training set shape: (80000, 11)
Testing set shape: (20000, 11)

=== Starting Optuna Hyperparameter Optimization for CatBoost with GPU ===


[I 2025-03-24 04:01:25,480] Trial 0 finished with value: 0.22911029728127127 and parameters: {'boosting_type': 'Ordered', 'bootstrap_type': 'Bernoulli', 'iterations': 1352, 'learning_rate': 0.10562899251590178, 'depth': 12, 'l2_leaf_reg': 4.10734111408579e-06, 'random_strength': 0.0030469501575328467, 'border_count': 249, 'subsample': 0.5278274700622245}. Best is trial 0 with value: 0.22911029728127127.
[I 2025-03-24 04:03:20,273] Trial 1 finished with value: 0.2311685746638357 and parameters: {'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'iterations': 5828, 'learning_rate': 0.1014859184937466, 'depth': 15, 'l2_leaf_reg': 0.12387719680060628, 'random_strength': 0.37807530682894075, 'border_count': 126, 'bagging_temperature': 4.56258813252531}. Best is trial 0 with value: 0.22911029728127127.
[I 2025-03-24 04:03:27,965] Trial 2 finished with value: 0.23390939428831867 and parameters: {'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'iterations': 4450, 'learning_ra

KeyboardInterrupt: 